# Install Dependencies

**Requirements:**
- stable-baselines3
- swig
- gymnasium[box2d]

> run it on Google Colab

In [1]:
!pip install stable-baselines3 -q
!pip install swig -q
!pip install gymnasium[box2d] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 34.2 MB/s eta 0:00:00


In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

import os
os.kill(os.getpid(), 9)

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Hit:13 https://ppa.launchpadcontent.n

session will reset automatically, so execute from the next cell in the next session.

## Hands-on gymnasium

In [1]:
import gymnasium
from pyvirtualdisplay import Display

from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from base64 import b64encode


1. To create environment: `gymnasium.make()`
2. To reset environment: `observation = env.reset()`
3. get random action: `action = env.action_space.sample()`

In [2]:
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [3]:
# create lunarlander v2 environment
env = gymnasium.make('LunarLander-v3', render_mode="rgb_array")
env = RecordVideo(env, './video')  # saves to /video folder

# reset environment to start a new episode
observation = env.reset()
observation

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


(array([ 0.00284863,  1.4220592 ,  0.28850856,  0.49506575, -0.00329395,
        -0.06535153,  0.        ,  0.        ], dtype=float32),
 {})

In [4]:
action = env.action_space.sample()
action

np.int64(1)

In [5]:
env.action_space, env.action_space.n

(Discrete(4), np.int64(4))

In [6]:
env.observation_space, env.observation_space.shape

(Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
   -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
   1.         1.       ], (8,), float32),
 (8,))

In [7]:
# take an action
observation, reward, terminated, truncated, info = env.step(action)

print("Action: ", action)
print("observation: ", observation)
print("reward: ", reward)
print("terminated: ", terminated)
print("truncated: ", truncated) #True if we hit the time limit (500 steps)
print("info: ", info)

Action:  1
observation:  [ 0.00562563  1.4326105   0.27915463  0.46894085 -0.00472587 -0.02864048
  0.          0.        ]
reward:  1.4966441560861778
terminated:  False
truncated:  False
info:  {}


In [8]:
episode_over = False
total_reward = 0

while not episode_over:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    episode_over = terminated

print(f"Episode finished! Total reward: {total_reward}")
env.close()

Episode finished! Total reward: -152.2448814037225


In [9]:
mp4 = open('./video/rl-video-episode-0.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=400 controls><source src="{data_url}" type="video/mp4"></video>')

## Train using Stable-Baselines3 library

> Stable Baselines3 (SB3) is an open-source Python library that provides reliable, high-quality implementations of Deep Reinforcement Learning (RL) algorithms built on PyTorch

In [39]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder

In [34]:
# Create 4 parallel environments using multiprocessing
env = make_vec_env('LunarLander-v3', n_envs=4, vec_env_cls=SubprocVecEnv)

In [46]:
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=0) #set verbose=1 if you want to save the training videos nad use the regular environment instead of vector environment

In [37]:
# Train it for 1,000 timesteps
model.learn(total_timesteps=1000000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 88.8     |
|    ep_rew_mean     | -160     |
| time/              |          |
|    fps             | 1260     |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 4096     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 86.5         |
|    ep_rew_mean          | -154         |
| time/                   |              |
|    fps                  | 980          |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 8192         |
| train/                  |              |
|    approx_kl            | 0.0037106096 |
|    clip_fraction        | 0.00476      |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | -0.0147      |
|    learning_r

In [31]:
# Save the model
model_name = "ppo-LunarLander-v2"
model.save(model_name)

In [29]:
eval_env = Monitor(gymnasium.make("LunarLander-v3", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=253.43 +/- 22.621740474282387


In [32]:
# --------- Evaluate -----------
env = gymnasium.make("LunarLander-v3", render_mode="rgb_array")
env = RecordVideo(env, video_folder="./video", episode_trigger=lambda x: x == 0)

obs, info = env.reset()
done = False
while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [33]:
mp4 = open('./video/rl-video-episode-0.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=400 controls><source src="{data_url}" type="video/mp4"></video>')